# 06 · Sample size gate, classification and reporting — Fetal death, MIMIC-IV (M3 · R3)

| Field | Content |
|---|---|
| Framework sequence | 01: S1 · S2 · S3 · G1 → 02: S4 → 03: S5 · S6 → R3 → 04: R3 · S2 · S3 · G1 · S4 → 05: S5 · S6 · S7 · G2 → **06: G3 · class · S11** |
| What it does | Applies gate G3 with the observed Cox-Snell R² of a cross-validated logistic regression (patient-grouped splits, imputation and scaling fitted on each training part) and the criteria of Riley et al.; only R² is computed (neither AUROC nor coefficients are looked at). With the result, it states the class of the model and what is reported (S11) |
| Stages and gates | G3 · class · S11 |
| Decisions | Class of M3; the reduced set is not applied (it is allowed only for a model that already meets G4) |
| Inputs | `antenatal_predictors_v1` (notebook 05), 7-day margin |
| Outputs | `data/outputs_06_sample_size_gate.txt` (aggregate output only) |
| Time | About a minute |
| Previous / next | `05_antenatal_predictors` / — |
| Spanish version | `06_sample_size_gate_es.ipynb` (same code) |

In [ ]:
# Setup: paths, language, presentation helpers and the database connection
import sys
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
# Language of the outputs: "es" in the Spanish notebook, "en" in the English one (the only line that differs).
LANG = "en"
import importlib
import libraries.i18n_es, libraries.i18n, libraries.presentation, libraries.sql_runner
for _mod in (libraries.i18n_es, libraries.i18n, libraries.presentation, libraries.sql_runner):
    importlib.reload(_mod)                       # a running kernel picks up changes of the libraries
from libraries.presentation import setup, tr, say, show, show_results, check, decision, notebook_summary, write_outputs
from libraries.sql_runner import run_script, ensure_table, table_exists
from libraries.database import get_connect_database
setup(LANG)

REBUILD = False      # True: rebuild the mimiciv tables of this notebook from their SQL scripts
SQL = ROOT / "sql"
OUT_FILE = ROOT / "data" / "outputs_06_sample_size_gate.txt"
engine = get_connect_database()
from libraries.antenatal_design import design_matrix, REDUCED_PARAMETERS
from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import IterativeImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedGroupKFold
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)  # 'penalty' renamed in recent scikit-learn

SEED = 20261119
BUFFER_DAYS = 7          # main information window
N_FOLDS = 5
S_TARGET = 0.9           # Riley criterion 1: expected shrinkage
DELTA = 0.05             # Riley criterion 2: optimism in Nagelkerke R2
MARGIN = 0.05            # Riley criterion 3: precision of the overall risk

## G3 · Sample size gate — data (G1: same population and events)

The 7-day margin of the predictor table. The notebook stops if the population or the events differ from gate G1.

| Result | What is done | In this model |
|---|---|---|
| 19,440 admissions and 129 events | On to the design matrix | Reproduced |
| They differ | It stops (`assert`): go back to notebooks 04 and 05 | — |

In [ ]:
df = pd.read_sql(
    "SELECT * FROM mimiciv.antenatal_predictors_v1 WHERE buffer_days = %(b)s",
    engine, params={"b": BUFFER_DAYS})

assert len(df) == 19440 and df["hadm_id"].is_unique, "population differs from gate G1"
assert int(df["label_main"].sum()) == 129, "events differ from gate G1"
show("G3 · Data of the gate (7-day margin)", pd.DataFrame([{"n_admissions": len(df), "n_patients": df["subject_id"].nunique(),
                                                             "n_strict_events": int(df["label_main"].sum())}]))

X = design_matrix(df)
y = df["label_main"].astype(int).to_numpy()
groups = df["subject_id"].to_numpy()
P = X.shape[1]
assert P == 27, P
show("G3 · Candidate parameters and missing values", pd.DataFrame([{"candidate parameters": P,
     **{f"missing {k}": int(v) for k, v in X.isna().sum()[X.isna().sum() > 0].items()}}]))

## G3 · Sample size gate — observed Cox-Snell R² (cross-validated) and criteria of Riley et al.

Out-of-fold predictions of an unpenalized logistic regression with the 27 parameters; 5 splits grouped by patient; multiple imputation and scaling fitted on the training part of each split. A negative R² means that, out of sample, the predictions are worse than giving every woman the same probability (the prevalence). Riley criteria: expected shrinkage of 0.9 or more; difference of 0.05 or less between apparent and adjusted Nagelkerke R²; overall risk within ±0.05.

| Result | What is done | In this model |
|---|---|---|
| The data meet the three criteria with the observed R² | G3 passes: on to S8 (model fitting) | — |
| They do not, with a positive R² | Reduced set (17), only for a model that meets G4, or next rung (R1) | — |
| Observed R² not positive | G3 does not pass with any number of parameters: the predictors carry no information on the outcome | R² = −0.00054: **does not pass** |

In [ ]:
# Out-of-fold predictions of an unpenalized logistic regression. Imputation and scaling are
# fitted on the training part of each split only; folds are grouped by patient.
def make_model():
    return make_pipeline(
        IterativeImputer(max_iter=10, random_state=SEED),
        StandardScaler(),
        LogisticRegression(penalty=None, max_iter=5000))

oof = np.full(len(y), np.nan)
cv = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
fold_rows = []
for k, (tr, te) in enumerate(cv.split(X, y, groups), 1):
    m = make_model().fit(X.iloc[tr], y[tr])
    oof[te] = m.predict_proba(X.iloc[te])[:, 1]
    fold_rows.append({"fold": k, "n_admissions": len(te), "n_events": int(y[te].sum())})
assert not np.isnan(oof).any()
show("G3 · Test part of each split", pd.DataFrame(fold_rows))

def loglik(y, p):
    p = np.clip(p, 1e-12, 1 - 1e-12)
    return float(np.sum(y * np.log(p) + (1 - y) * np.log(1 - p)))

n, E = len(y), int(y.sum())
phi = E / n
L0 = n * (phi * np.log(phi) + (1 - phi) * np.log(1 - phi))   # intercept-only model
L1 = loglik(y, oof)
R2_cs = 1 - np.exp(2 * (L0 - L1) / n)
R2_max = 1 - np.exp(2 * L0 / n)

# Riley et al. (BMJ 2020) criteria for a binary outcome
def riley(p, R2, R2max, phi):
    if R2 <= 0:
        return None
    n1 = p / ((S_TARGET - 1) * np.log(1 - R2 / S_TARGET))
    S2 = R2 / (R2 + DELTA * R2max)
    n2 = p / ((S2 - 1) * np.log(1 - R2 / S2))
    n3 = (1.96 / MARGIN) ** 2 * phi * (1 - phi)
    S = max(S_TARGET, S2)
    p_max = int(np.floor(n * (S - 1) * np.log(1 - R2 / S)))   # parameters the data support
    n_req = int(np.ceil(max(n1, n2, n3)))
    return dict(n1=n1, n2=n2, n3=n3, S2=S2, n_required=n_req,
                events_required=int(np.ceil(n_req * phi)), p_supported=p_max)

res = riley(P, R2_cs, R2_max, phi)
if res is None:
    gate = "NOT PASSED (cross-validated R2 is not positive)"
else:
    gate = "PASSED" if n >= res["n_required"] else "NOT PASSED"

# smallest R2 with which the n available admissions would support p parameters (criterion 1, shrinkage 0.9)
r2_needed = lambda p: S_TARGET * (1 - np.exp(p / (n * (S_TARGET - 1))))
show("G3 · Observed Cox-Snell R2 and minimum R2 for the candidate parameters", pd.DataFrame([
    ("prevalence", round(phi, 4)), ("maximum Cox-Snell R2", round(R2_max, 4)),
    ("observed Cox-Snell R2 (cross-validated)", round(R2_cs, 5)),
    ("observed R2, % of the maximum", round(100 * R2_cs / R2_max, 1)),
    (f"minimum R2 for {P} parameters", round(r2_needed(P), 4)),
    (f"minimum R2 for {REDUCED_PARAMETERS} parameters (reduced set)", round(r2_needed(REDUCED_PARAMETERS), 4)),
    ("gate G3", gate)], columns=["indicator", "value"]))
if res is not None:
    show("G3 · Criteria of Riley et al.", pd.Series(res, name="value").to_frame(), index=True)

## G3 · Decision of the gate, class of the model and reporting (S11)

The reduced set is allowed only for a model that already meets gate G4, and with a negative R² it would not pass either. Return R3 was already used and the outcome ladder of MIMIC-IV has no more frequent rung for fetal death in this route, so the model stays as a **feasibility analysis**. Stages S8 to S10 and gate G4 are not carried out.

| Result | What is done | In this model |
|---|---|---|
| G3 does not pass and no return or rung is left | Class: feasibility; what is reported is the audit, the reason of the return and the result of G3 (S11) | **Feasibility** |
| G3 passes | On to S8 (model fitting) | — |

In [ ]:
# Decision of gate G3 and class of M3
passed = gate == "PASSED"
decision("G3 · Sample size gate", [
    ("observed Cox-Snell R2 with 27 parameters", f"{R2_cs:.5f} ({100 * R2_cs / R2_max:.1f}% of the maximum)",
     "passes: on to S8" if passed else "does not pass"),
    ("reduced set of 17 parameters", f"it would need R2 of at least {r2_needed(REDUCED_PARAMETERS):.4f}",
     "not applied: allowed only for a model that meets G4, and with this R2 it would not pass either"),
    ("returns", "R3 already used (admission + 6 h -> antenatal)", "no more frequent rung for fetal death in MIMIC-IV")])
klass = "FEASIBILITY" if not passed else "goes on to S8"
decision("Class of the model and reporting (S11)", [
    ("M3 at admission + 6 h", "stopped at S6: more than half of the deaths already present", "return R3 (notebook 03)"),
    ("M3 · R3 antenatal", "stopped at G3: observed R2 not positive", klass),
    ("stages S8 to S10 and gate G4", "not carried out", "—"),
    ("reporting (S11)", "count audit, reason of the return and result of G3", "with the information MIMIC-IV records before the admission, fetal death cannot be predicted")])

## Notebook summary

Main indicators of this notebook and the next step.

In [ ]:
# Summary of notebook 06 and of model M3
notebook_summary([
    ("admissions / strict events", lambda: f"{n:,} / {E}"),
    ("candidate parameters", lambda: P),
    ("observed Cox-Snell R2 (cross-validated)", lambda: f"{R2_cs:.5f} ({100 * R2_cs / R2_max:.1f}% of the maximum {R2_max:.4f})"),
    ("minimum R2 for 27 / 17 parameters", lambda: f"{r2_needed(27):.4f} / {r2_needed(17):.4f}"),
    ("gate G3", lambda: gate),
    ("class of M3", lambda: klass),
    ("path of M3", "S1-S6 at admission + 6 h -> return R3 -> S2-S7 and G1, G2 antenatal -> G3 does not pass"),
])
lines_gate = [
    "Gate G3 (sample size), antenatal model; aggregate output only",
    f"buffer_days={BUFFER_DAYS}; folds={N_FOLDS} grouped by patient; seed={SEED}",
    f"admissions={n}; events={E}; prevalence={phi:.4f}; candidate_parameters={P}",
    f"R2_cs_max={R2_max:.4f}; R2_cs_observed_cv={R2_cs:.5f} ({100 * R2_cs / R2_max:.1f}% of max)",
]
if res is not None:
    lines_gate += [f"n_required={res['n_required']} (criterion1={res['n1']:.0f}, criterion2={res['n2']:.0f}, criterion3={res['n3']:.0f}); "
                   f"events_required={res['events_required']}; parameters_supported={res['p_supported']}"]
lines_gate += [f"gate={gate}"]
write_outputs(OUT_FILE, "\n".join(lines_gate))